# LIBERO Dataset Episode Inspector

Quickly inspect every episode in a LIBERO HDF5 dataset. The notebook is meant for debugging task-specific dataset issues, especially when BC training metrics look good but eval success is poor.

It does not change training preprocessing or write into the dataset. Outputs go to `visualization/results/dataset_episode_inspection/`.

In [ ]:
from __future__ import annotations

import os
import sys
import math
import json
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "visualization" else Path.cwd()
LIBERO_ROOT = Path(os.environ.get("LIBERO_REPO_PATH", REPO_ROOT / "LIBERO")).expanduser().resolve()

SUITE = "libero_spatial"
DATASET_NAME = os.environ.get("LIBERO_DATASET_NAME", "libero_spatial_256_from_rlds_reverted")
DATASET_DIR = LIBERO_ROOT / "libero" / "datasets" / DATASET_NAME

# Set TASK_IDS=None for all tasks, or e.g. TASK_IDS=[5, 9] for focused comparison.
TASK_IDS = None
MAX_DEMOS_PER_TASK = None

# Visual settings.
ROTATE_IMAGES_AS_ROLLOUT = True
FRAMES_PER_DEMO = 5
THUMB_WIDTH = 160
LABEL_HEIGHT = 28

OUTPUT_DIR = REPO_ROOT / "visualization" / "results" / "dataset_episode_inspection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT={REPO_ROOT}")
print(f"LIBERO_ROOT={LIBERO_ROOT}")
print(f"DATASET_DIR={DATASET_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
assert LIBERO_ROOT.is_dir(), f"Missing LIBERO root: {LIBERO_ROOT}"
assert DATASET_DIR.is_dir(), f"Missing dataset dir: {DATASET_DIR}"

## Build a Non-Interactive LIBERO Config

LIBERO can prompt on import if `LIBERO_CONFIG_PATH` is unset. This cell writes a small temporary config under the output directory so the notebook stays non-interactive.

In [ ]:
CONFIG_DIR = OUTPUT_DIR / "libero_config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
config = {
    "benchmark_root": str(LIBERO_ROOT / "libero" / "libero"),
    "bddl_files": str(LIBERO_ROOT / "libero" / "libero" / "bddl_files"),
    "init_states": str(LIBERO_ROOT / "libero" / "libero" / "init_files"),
    "datasets": str(LIBERO_ROOT / "libero" / "datasets"),
    "assets": str(LIBERO_ROOT / "libero" / "libero" / "assets"),
}
(CONFIG_DIR / "config.yaml").write_text("\n".join(f"{k}: {v}" for k, v in config.items()) + "\n")
os.environ["LIBERO_CONFIG_PATH"] = str(CONFIG_DIR)
sys.path.insert(0, str(LIBERO_ROOT))

from libero.libero.benchmark import get_benchmark

benchmark = get_benchmark(SUITE)()
num_tasks = benchmark.get_num_tasks()
if TASK_IDS is None:
    TASK_IDS = list(range(num_tasks))

task_rows = []
for task_id in TASK_IDS:
    task = benchmark.get_task(task_id)
    task_rows.append({
        "task_id": task_id,
        "task_name": task.name,
        "language": task.language,
        "demo_file": f"{task.name}_demo.hdf5",
        "init_states_file": task.init_states_file,
        "exists": (DATASET_DIR / f"{task.name}_demo.hdf5").is_file(),
    })

task_df = pd.DataFrame(task_rows)
display(task_df)
task_df.to_csv(OUTPUT_DIR / "task_file_mapping.csv", index=False)

## Episode Summary Table

This scans every selected HDF5 episode and records lengths, image shapes, state availability, and action/gripper summaries.

In [ ]:
def task_file(task_id: int) -> Path:
    task = benchmark.get_task(task_id)
    return DATASET_DIR / f"{task.name}_demo.hdf5"


def demo_names_for_file(path: Path, max_demos: int | None = None) -> list[str]:
    with h5py.File(path, "r") as f:
        names = sorted(f["data"].keys())
    return names[:max_demos] if max_demos is not None else names


def summarize_episode(path: Path, task_id: int, demo_name: str) -> dict:
    with h5py.File(path, "r") as f:
        demo = f["data"][demo_name]
        actions = np.asarray(demo["actions"])
        obs = demo.get("obs", None)
        img_shape = None
        wrist_shape = None
        if obs is not None and "agentview_rgb" in obs:
            img_shape = tuple(obs["agentview_rgb"].shape)
        if obs is not None and "eye_in_hand_rgb" in obs:
            wrist_shape = tuple(obs["eye_in_hand_rgb"].shape)
        has_states = "states" in demo
        state_shape = tuple(demo["states"].shape) if has_states else None
        attrs = {k: demo.attrs[k] for k in demo.attrs.keys()}

    flat = actions.reshape(-1, actions.shape[-1])
    grip = flat[:, -1]
    env_grip = np.sign(2 * grip - 1) * -1.0
    return {
        "task_id": task_id,
        "demo": demo_name,
        "path": str(path),
        "length": int(actions.shape[0]),
        "action_dim": int(actions.shape[-1]),
        "image_shape": img_shape,
        "wrist_shape": wrist_shape,
        "has_states": has_states,
        "state_shape": state_shape,
        "action_min": flat.min(axis=0).round(5).tolist(),
        "action_max": flat.max(axis=0).round(5).tolist(),
        "action_mean": flat.mean(axis=0).round(5).tolist(),
        "action_std": flat.std(axis=0).round(5).tolist(),
        "gripper_min": float(grip.min()),
        "gripper_max": float(grip.max()),
        "gripper_mean": float(grip.mean()),
        "gripper_frac_neg": float(np.mean(grip < 0)),
        "gripper_frac_zero": float(np.mean(grip == 0)),
        "gripper_frac_one": float(np.mean(grip == 1)),
        "env_gripper_unique": sorted(np.unique(env_grip).astype(float).tolist()),
        "attrs": attrs,
    }


rows = []
for task_id in TASK_IDS:
    path = task_file(task_id)
    if not path.is_file():
        print(f"Missing task {task_id}: {path}")
        continue
    for demo_name in demo_names_for_file(path, MAX_DEMOS_PER_TASK):
        rows.append(summarize_episode(path, task_id, demo_name))

episodes_df = pd.DataFrame(rows)
episodes_df.to_csv(OUTPUT_DIR / "episode_summary.csv", index=False)
print(f"episodes={len(episodes_df)} saved={OUTPUT_DIR / 'episode_summary.csv'}")
display(episodes_df.head(20))

In [ ]:
task_summary = (
    episodes_df.groupby("task_id")
    .agg(
        demos=("demo", "count"),
        length_min=("length", "min"),
        length_mean=("length", "mean"),
        length_max=("length", "max"),
        gripper_mean=("gripper_mean", "mean"),
        gripper_frac_neg=("gripper_frac_neg", "mean"),
        gripper_frac_zero=("gripper_frac_zero", "mean"),
        gripper_frac_one=("gripper_frac_one", "mean"),
    )
    .reset_index()
)
task_summary = task_summary.merge(task_df[["task_id", "task_name", "language"]], on="task_id", how="left")
task_summary.to_csv(OUTPUT_DIR / "task_summary.csv", index=False)
display(task_summary)

## Length and Action Distribution Plots

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
episodes_df.boxplot(column="length", by="task_id", ax=ax)
ax.set_title("Episode Length by Task")
ax.set_xlabel("Task ID")
ax.set_ylabel("Steps")
fig.suptitle("")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "episode_lengths_by_task.png", dpi=180)
plt.show()

def read_actions_for_task(task_id: int) -> np.ndarray:
    path = task_file(task_id)
    parts = []
    for demo_name in demo_names_for_file(path, MAX_DEMOS_PER_TASK):
        with h5py.File(path, "r") as f:
            parts.append(np.asarray(f["data"][demo_name]["actions"], dtype=np.float32))
    return np.concatenate(parts, axis=0)

for task_id in TASK_IDS:
    path = task_file(task_id)
    if not path.is_file():
        continue
    actions = read_actions_for_task(task_id)
    flat = actions.reshape(-1, actions.shape[-1])
    fig, axes = plt.subplots(1, 7, figsize=(16, 3), sharey=False)
    for dim, ax in enumerate(axes):
        ax.hist(flat[:, dim], bins=50, color="#4C78A8", alpha=0.85)
        ax.set_title(f"a{dim}")
        ax.grid(alpha=0.2)
    fig.suptitle(f"Task {task_id}: action distributions")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"task_{task_id}_action_histograms.png", dpi=180)
    plt.show()

## Contact Sheets for Every Episode

Each row is one demo. Columns are evenly spaced frames from that demo. Set `TASK_IDS=[5]` in the first cell if you only want task 5.

In [ ]:
def _frame_indices(length: int, n: int) -> list[int]:
    if length <= 0:
        return []
    if n <= 1:
        return [0]
    return sorted(set(np.linspace(0, length - 1, n).round().astype(int).tolist()))


def _to_thumb(img: np.ndarray, width: int) -> Image.Image:
    img = np.asarray(img)
    if ROTATE_IMAGES_AS_ROLLOUT:
        img = np.ascontiguousarray(img[::-1, ::-1])
    pil = Image.fromarray(img.astype(np.uint8))
    h = max(1, int(round(width * pil.height / pil.width)))
    return pil.resize((width, h), Image.BILINEAR)


def make_contact_sheet(task_id: int, camera_key: str = "agentview_rgb") -> Path:
    path = task_file(task_id)
    demos = demo_names_for_file(path, MAX_DEMOS_PER_TASK)
    rows = []
    for demo_name in demos:
        with h5py.File(path, "r") as f:
            demo = f["data"][demo_name]
            if "obs" not in demo or camera_key not in demo["obs"]:
                continue
            images = demo["obs"][camera_key]
            actions = np.asarray(demo["actions"])
            idxs = _frame_indices(len(images), FRAMES_PER_DEMO)
            thumbs = [_to_thumb(images[i], THUMB_WIDTH) for i in idxs]
            rows.append((demo_name, len(images), idxs, actions[:, -1], thumbs))

    if not rows:
        raise RuntimeError(f"No image rows for task {task_id} camera={camera_key}")

    thumb_h = rows[0][4][0].height
    label_w = 210
    sheet_w = label_w + FRAMES_PER_DEMO * THUMB_WIDTH
    sheet_h = len(rows) * (thumb_h + LABEL_HEIGHT)
    sheet = Image.new("RGB", (sheet_w, sheet_h), "white")
    draw = ImageDraw.Draw(sheet)

    y = 0
    for demo_name, length, idxs, grip, thumbs in rows:
        text = f"{demo_name}  len={length}\ngrip {grip.min():.1f}..{grip.max():.1f}"
        draw.text((6, y + 4), text, fill=(0, 0, 0))
        for col, thumb in enumerate(thumbs):
            x = label_w + col * THUMB_WIDTH
            sheet.paste(thumb, (x, y))
            draw.text((x + 4, y + thumb.height + 4), f"t={idxs[col]}", fill=(0, 0, 0))
        y += thumb_h + LABEL_HEIGHT

    out = OUTPUT_DIR / f"task_{task_id}_{camera_key}_contact_sheet.png"
    sheet.save(out)
    return out


contact_paths = []
for task_id in TASK_IDS:
    if task_file(task_id).is_file():
        contact_paths.append(make_contact_sheet(task_id, "agentview_rgb"))

print("Wrote contact sheets:")
for p in contact_paths:
    print(p)

In [ ]:
# Preview a single generated contact sheet inline.
TASK_TO_PREVIEW = 5 if 5 in TASK_IDS else TASK_IDS[0]
preview_path = OUTPUT_DIR / f"task_{TASK_TO_PREVIEW}_agentview_rgb_contact_sheet.png"
display(Image.open(preview_path))

## Single-Episode Frame Strip

Use this when one episode looks suspicious in the contact sheet.

In [ ]:
TASK_TO_VIEW = 5 if 5 in TASK_IDS else TASK_IDS[0]
DEMO_TO_VIEW = "demo_0"
NUM_FRAMES_TO_VIEW = 16

path = task_file(TASK_TO_VIEW)
with h5py.File(path, "r") as f:
    demo = f["data"][DEMO_TO_VIEW]
    images = demo["obs"]["agentview_rgb"]
    actions = np.asarray(demo["actions"])
    idxs = _frame_indices(len(images), NUM_FRAMES_TO_VIEW)

cols = min(8, len(idxs))
rows_n = math.ceil(len(idxs) / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 2.2, rows_n * 2.2))
axes = np.asarray(axes).reshape(-1)
for ax, idx in zip(axes, idxs):
    img = np.asarray(images[idx])
    if ROTATE_IMAGES_AS_ROLLOUT:
        img = img[::-1, ::-1]
    ax.imshow(img)
    ax.set_title(f"t={idx}\ngrip={actions[idx, -1]:.1f}", fontsize=9)
    ax.axis("off")
for ax in axes[len(idxs):]:
    ax.axis("off")
fig.suptitle(f"Task {TASK_TO_VIEW} {DEMO_TO_VIEW}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"task_{TASK_TO_VIEW}_{DEMO_TO_VIEW}_frame_strip.png", dpi=180)
plt.show()

## What to Look For

- Task 5 file/language mismatch relative to the benchmark task table.
- Very unusual episode lengths compared with other tasks.
- Demo images not matching the task language, or object placement visibly wrong.
- A task where gripper statistics collapse to one command.
- Missing `states` for task 5 while other tasks have them, or state/action length mismatches.

If these static checks look normal, the next decisive test is an environment replay check: reset LIBERO to demo states and execute the stored task 5 actions. That tells us whether the task 5 HDF5 itself is replayable in the current simulator.